In [ ]:
!pip uninstall -y torch torchtext torchvision torchaudio && pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121
!pip install torchtext --no-cache-dir
!pip install einops -q

In [11]:
import torch
import torch.nn as nn
import math

In [21]:
img_test =  torch.randint(0, 2, (4,1 ,50, 200))
img_test = img_test.float()

In [ ]:
print(img_test)

In [ ]:
from google.colab import drive
drive.mount("/content/drive/")

In [ ]:
data_set = torch.load("/content/drive/MyDrive/Group Project 2025-2026/training_data.pt")

In [ ]:
img_test1 = data_set['images'][0]
print(img_test1)
img_test1 = img_test1.unsqueeze(0)
print(img_test1.shape)

In [63]:
class Cnn_layer(nn.Module):
  def __init__(self, input_channel, height, width, embedding_dim ,max_len = 5000 ):
    super().__init__()
    self.feature = nn.Sequential(
        #Block 1
        nn.Conv2d(input_channel, 8, kernel_size=3,stride=1, padding=1),
        nn.BatchNorm2d(8),
        nn.LeakyReLU(),
        nn.MaxPool2d(2),
        nn.Dropout2d(0.2),
        #Block 2
        nn.Conv2d(8, 16, kernel_size=3,stride=1, padding=1),
        nn.BatchNorm2d(16),
        nn.LeakyReLU(),
        nn.MaxPool2d(2),
        nn.Dropout2d(0.2),
        #Block 3
        nn.Conv2d(16, 32, kernel_size=3,stride=1, padding=1),
        nn.BatchNorm2d(32),
        nn.LeakyReLU(),
        nn.MaxPool2d(2),
        nn.Dropout2d(0.2),
        #Block 4
        nn.Conv2d(32, 64, kernel_size=3,stride=1, padding=1),
        nn.BatchNorm2d(64),
        nn.LeakyReLU(),

        nn.Dropout2d(0.2),
        #Block 5
        nn.Conv2d(64, 128, kernel_size=3,stride=1, padding=1),
        nn.BatchNorm2d(128),
        nn.LeakyReLU(),

        nn.Dropout2d(0.2)
    )
    self.projection = nn.Linear(128, embedding_dim)
    self.pos_embed = nn.Parameter(torch.randn(1, max_len, embedding_dim))
  def forward(self,x):
  # B, C, H, W
    x = self.feature(x)
    x = torch.flatten(x, start_dim=2)
    x = x.permute(0,2,1)
    x = self.projection(x)
    seq_len = x.shape[1]
    x = x + self.pos_embed[:, :seq_len, :]

    return x

In [ ]:
te = Cnn_layer(1,50,200,256)
x = te.forward(img_test)
print(x.shape)

In [45]:
class MultiheadAttention(nn.Module):
  def __init__(self, embedding_dim, num_head:int=4 , dropout: float = 0) -> None:
     super().__init__()
     self.layer_norm=nn.LayerNorm(embedding_dim)
     self.mha = nn.MultiheadAttention(embed_dim=embedding_dim,
                                      num_heads=num_head,
                                      dropout=dropout,
                                      batch_first=True)
  def forward(self,x):
    x_norm = self.layer_norm(x)
    x_attention,_ = self.mha(query = x_norm,
                           key = x_norm,
                           value = x_norm,
                           need_weights = False)
    return  x + x_attention

In [46]:
class MLP_Block(nn.Module):
  def __init__(self, embedding_dim,mlp_size: int = 1024 ,dropout: float = 0.2) :
     super().__init__()
     self.layer_norm=nn.LayerNorm(embedding_dim)
     self.mlp = nn.Sequential(nn.Linear(embedding_dim,mlp_size),
                              nn.GELU(),
                              nn.Dropout(dropout),
                              nn.Linear(mlp_size,embedding_dim),
                              nn.Dropout(dropout))
  def forward(self,x):
    x_norm = self.layer_norm(x)
    x_mlp = self.mlp(x_norm)
    return x + x_mlp

In [47]:
class TransformerEncoderLayer(nn.Module):
  def __init__(self, embedding_dim, num_head:int=4 , attn_dropout: float = 0 , mlp_size: int =1024 , mlp_dropout: float = 0.2 ):
     super().__init__()
     self.attn = MultiheadAttention(embedding_dim,num_head,attn_dropout)
     self.mlp = MLP_Block(embedding_dim,mlp_size,mlp_dropout)
  def forward(self,x):
    x_attn = self.attn(x)
    x_mlp = self.mlp(x_attn)
    return x_mlp

In [48]:
class TransformerEncoder(nn.Module):
    def __init__(self, embedding_dim, num_layers, num_head=4, attn_dropout=0, mlp_size=1024, mlp_dropout=0.2):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(
                embedding_dim=embedding_dim,
                num_head=num_head,
                mlp_size=mlp_size,
                attn_dropout=attn_dropout,
                mlp_dropout=mlp_dropout
            )
            for _ in range(num_layers)
        ])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

In [65]:
class CvT(nn.Module):
  def __init__(self,  input_channel, height, width, embedding_dim,num_layer ):
     super().__init__()
     self.cnn = Cnn_layer(input_channel, height, width, embedding_dim)

     self.transformer= TransformerEncoder(embedding_dim ,num_layer, num_head=4, attn_dropout=0, mlp_size=1024, mlp_dropout=0.2)
  def forward(self,x):
    out1 = self.cnn(x)
    out2 = self.transformer(out1)
    return out2

In [ ]:
cvt = CvT(1,50,200,256,4)
tr= cvt.forward(img_test)
print(tr.shape)
print(tr)